In [1]:
using Clapeyron, Metaheuristics, Printf

In [2]:
#Hfus dan Tm masukin ke csv yh jgn lupa

like_parameter = """Clapeyron Database File
PCSAFT Like Parameters [csvtype = like,grouptype = PCSAFT]
species,Mw,segment,sigma,epsilon,n_H,n_e
glycine,75.07,4.8495,2.327,216.96,2,2
water,18.02,1.2047,2.801457,353.94,1,1
"""

unlike_parameter = """Clapeyron Database File
PCSAFT Unlike Parameters [csvtype = unlike,grouptype = PCSAFT]
species1,species2,k
water,glycine,-0.0612
"""

#jgn lupa combing rules yh
assoc_parameter = """Clapeyron Database File
PCSAFT Assoc Parameters [csvtype = assoc,grouptype = PCSAFT]
species1,site1,species2,site2,epsilon_assoc,bondvol
water,H,water,e,2425.67,0.045
glycine,H,glycine,e,2598.0600,0.0393
water,H,glycine,e,2511.865,0.041514793
water,e,glycine,H,2511.865,0.041514793
"""
components = ["water", "glycine"]
model = CompositeModel(components;
                       fluid = PCSAFT,
                       solid = SolidHfus,
                       fluid_userlocations = [like_parameter, unlike_parameter, assoc_parameter])

println(model.fluid.params.epsilon.values)
println(model.fluid.params.sigma.values)
println("======================")
println(model.fluid.params.epsilon_assoc.values)
println(model.fluid.params.bondvol.values)
println("kij = ", (1  - ((model.fluid.params.epsilon.values[2])/(sqrt(model.fluid.params.epsilon.values[1] * model.fluid.params.epsilon.values[4])))))

[353.94 294.07079841359604; 294.07079841359604 216.96]
[2.8014570000000003e-10 2.5642285e-10; 2.5642285e-10 2.327e-10]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 2511.865, 2511.865, 2598.06]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.041514793, 0.041514793, 0.0393]
kij = -0.06119999999999992


In [3]:
# Run this ONCE to fix your CSV files
function fix_line_endings(filename)
    content = read(filename, String)
    fixed = replace(content, "\r\n" => "\n")
    write(filename, fixed)
    println("Fixed: $filename")
end

fix_line_endings("gamma_glycine.csv")
fix_line_endings("rhol_glycine.csv")

Fixed: gamma_glycine.csv
Fixed: rhol_glycine.csv


In [4]:
#SOLUTION DENSITY

Mw_water = model.fluid.params.Mw[1] / 1000
Mw_aa = model.fluid.params.Mw[2] / 1000

println("MW Solute  : ", Mw_aa, " kg/m3")
println("MW solvent : ", Mw_water, " kg/m3")

function molality_to_z(m::Float64, Mw_solute::Float64)
    # m mol solute per kg water
    n_solute = m
    n_water  = 1.0 / Mw_water   # mol water per kg water = 55.51
    x_water  = n_water / (n_water + n_solute)
    x_solute = n_solute / (n_water + n_solute)
    return [x_water, x_solute]   # [water, amino acid]
end

function solution_density(model::EoSModel, m::Float64)
    z   = molality_to_z(m, Mw_aa)
    T   = 298.15
    p   = 101325.0
    V   = volume(model, p, T, z; phase=:liquid)        # m³/mol mixture
    Mw_mix = z[1]*Mw_water + z[2]*Mw_aa               # kg/mol
    return Mw_mix / V                                   # kg/m³
end

MW Solute  : 0.07507 kg/m3
MW solvent : 0.01802 kg/m3


solution_density (generic function with 1 method)

In [5]:
#ACTIVITY COEFFICIENT

function chem_activity_coeff(model::EoSModel, m::Float64)
    z     = molality_to_z(m, 75.07)
    z_inf = molality_to_z(1e-10, 75.07)   # very dilute reference
    p     = 101325.0
    T     = 298.15
    RT    = Rgas(model) * T

    # get chemical potentials directly
    chem_pot     = chemical_potential(model.fluid, p, T, z;     phase=:liquid)
    chem_pot_inf = chemical_potential(model.fluid, p, T, z_inf; phase=:liquid)

    # γ dari persamaan chemical potential
    gamma_star_x = exp((chem_pot[2] - chem_pot_inf[2]) / RT) * (z_inf[2] / z[2])

    # convert to molality basis (Kuramochi eq 5)
    #println("exp((μ[2] - μ_inf[2]) / RT) :", exp((μ[2] - μ_inf[2]) / RT))
    #println("m :",m)
    #println("z_inf :",z_inf)
    #println("z[2]: ",z[2])
    #println("z_inf[2] / z[2] :",z_inf[2] / z[2])
    return gamma_star_x / (1.0 + Mw_water * m)
end

chem_activity_coeff (generic function with 1 method)

In [6]:
toestimate = [
    Dict(
        :param => :epsilon,
        :indices => (2,2), #kij
        :lower => 200.,
        :upper => 400.,
        :guess => 300.
    ),
    Dict(
        :param => :sigma,
        :indices => (2,2),
        :factor => 1e-10,
        :lower => 2.0,
        :upper => 4.0,
        :guess => 3.
    ),
    Dict(
        :param   => :epsilon_assoc,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 2000.0,
        :upper   => 3000.0,
        :guess   => 2500.0
    ),
    Dict(
        :param   => :bondvol,
        :indices => 4,
        #:cross_assoc => true,
        :lower   => 0.03,
        :upper   => 0.5,
        :guess   => 0.01
    ),
]

4-element Vector{Dict{Symbol, Any}}:
 Dict(:upper => 400.0, :param => :epsilon, :indices => (2, 2), :guess => 300.0, :lower => 200.0)
 Dict(:factor => 1.0e-10, :param => :sigma, :indices => (2, 2), :upper => 4.0, :lower => 2.0, :guess => 3.0)
 Dict(:upper => 3000.0, :param => :epsilon_assoc, :indices => 4, :guess => 2500.0, :lower => 2000.0)
 Dict(:upper => 0.5, :param => :bondvol, :indices => 4, :guess => 0.01, :lower => 0.03)

In [7]:
estimator, objective, x0, upper, lower = Estimation(
    model,
    toestimate,
    [
        "rhol_glycine.csv",
        "gamma_glycine.csv",
    ]
)
 
println("Initial objective value: ", objective(x0))
println(x0)

Initial objective value: 0.19320745549483603
[300.0, 3.0, 2500.0, 0.01]


In [8]:
method = ECA(; options = Options(iterations = 10000, seed = 42))
 
params_opt, model_opt = optimize(objective, estimator, method)

([207.07076908385957, 2.1512718466503644, 2410.1881292411545, 0.0805618263928317], CompositeModel{PCSAFT{BasicIdeal, Float64}, SolidHfus}("water", "glycine"))

In [9]:
println("\n=== Optimized PC-SAFT parameters for glycine ===")
println("  segment (m)  : ", model_opt.fluid.params.segment[2])
println("  sigma   (m)  : ", model_opt.fluid.params.sigma[2,2])
println("  epsilon (K1)  : ", model_opt.fluid.params.epsilon.values)
println("  epsilon_assoc  : ", model_opt.fluid.params.epsilon_assoc)
println("  bondvol  : ", model_opt.fluid.params.bondvol)
println("==================")
println(model_opt.fluid.params.epsilon.values)
println(model_opt.fluid.params.sigma.values)
println("kij = ", (1  - ((model_opt.fluid.params.epsilon.values[2])/(sqrt(model_opt.fluid.params.epsilon.values[1] * model_opt.fluid.params.epsilon.values[4])))))
println("======================")
println(model_opt.fluid.params.epsilon_assoc.values)
println(model_opt.fluid.params.bondvol.values)



=== Optimized PC-SAFT parameters for glycine ===
  segment (m)  : 4.8495
  sigma   (m)  : 2.1512718466503645e-10
  epsilon (K1)  : [353.94 294.07079841359604; 294.07079841359604 207.07076908385957]
  epsilon_assoc  : AssocParam{Float64}("epsilon_assoc")[2425.67, 2511.865, 2511.865, 2410.1881292411545]
  bondvol  : AssocParam{Float64}("bondvol")[0.045, 0.041514793, 0.041514793, 0.0805618263928317]
[353.94 294.07079841359604; 294.07079841359604 207.07076908385957]
[2.8014570000000003e-10 2.4763644233251827e-10; 2.4763644233251827e-10 2.1512718466503645e-10]
kij = -0.08624472173396946
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[2425.67, 2511.865, 2511.865, 2410.1881292411545]
Clapeyron.Compressed4DMatrix{Float64, Vector{Float64}}[0.045, 0.041514793, 0.041514793, 0.0805618263928317]


In [10]:
using CSV, DataFrames, Printf

function calculate_AAD(model, csv_file, property_func)
    df = CSV.read(csv_file, DataFrame, comment="#", skipto=4)
    
    input_col  = names(df)[1]          # first column = input (T)
    output_col = names(df)[2]          # second column = out_xxx (experimental)
    
    inputs   = df[!, input_col]
    exp_vals = df[!, output_col]
    
    println("\n=== AAD: $csv_file ===")
    @printf("%-10s  %-12s  %-12s  %-8s\n", input_col, "exp", "calc", "ARD%")
    
    errors = Float64[]
    for (i, x) in enumerate(inputs)
        calc = property_func(model, x)
        err  = abs(calc - exp_vals[i]) / abs(exp_vals[i]) * 100
        push!(errors, err)
        @printf("%-10.4f  %-12.6f  %-12.6f  %-8.4f\n", x, exp_vals[i], calc, err)
    end
    
    aard = sum(errors) / length(errors)
    @printf("AARD = %.4f%%\n", aard)
    return aard
end

calculate_AAD (generic function with 1 method)

In [11]:
aard_p   = calculate_AAD(model_opt, "rhol_glycine.csv", solution_density)


=== AAD: rhol_glycine.csv ===


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


Clapeyron Estimator  exp           calc          ARD%    
0.0000      997.045000    992.536576    0.4522  
0.0310      998.018000    993.806186    0.4220  
0.0707      999.293000    995.430536    0.3865  
0.1080      1000.403000   996.951328    0.3450  
0.1551      1001.899000   998.868034    0.3025  
0.2039      1003.401000   1000.848423   0.2544  
0.2732      1005.532000   1003.651155   0.1870  
0.3137      1006.753000   1005.283925   0.1459  
0.3639      1008.283000   1007.302457   0.0972  
0.4048      1009.618000   1008.942714   0.0669  
0.4937      1012.146000   1012.494627   0.0344  
AARD = 0.2449%


0.24492810356132474

In [12]:
aard_p   = calculate_AAD(model_opt, "gamma_glycine.csv", chem_activity_coeff)


=== AAD: gamma_glycine.csv ===
Clapeyron Estimator  exp           calc          ARD%    
0.1006      0.985200      0.983130      0.2101  
0.1485      0.978300      0.975514      0.2848  
0.2507      0.967300      0.960108      0.7435  
0.3011      0.956700      0.952914      0.3957  
0.3509      0.949800      0.946056      0.3942  
0.4024      0.942800      0.939215      0.3803  
0.4493      0.936500      0.933200      0.3524  
0.5123      0.928100      0.925432      0.2875  
0.9976      0.867500      0.876102      0.9916  
AARD = 0.4489%


┌ Warning: thread = 1 warning: parsed expected 1 columns, but didn't reach end of line around data row: 1. Parsing extra columns and widening final columnset
└ @ CSV C:\Users\sutha\.julia\packages\CSV\XLcqT\src\file.jl:593


0.44889433571135995